# Singular Spectrum Analysis (SSA) Formula

This notebook provides the pure mathematical formulation of the **Singular Spectrum Analysis (SSA)** algorithm.

---

### 1. Embedding (Trajectory Matrix)
Maps the time series $y = (y_1, \dots, y_N)$ into an $L \times K$ trajectory matrix $X$:

$$X_{ij} = y_{i+j-1} \quad (1 \le i \le L, \ 1 \le j \le K)$$

### 2. Singular Value Decomposition (SVD)
Decomposes the matrix $X$ into elementary components:

$$X = \sum_{i=1}^{L} \sigma_i U_i V_i^T$$

### 3. Diagonal Averaging (Hankelization)
Transforms the filtered matrix $\tilde{X}$ back into a 1D series $\tilde{y}_k$ by averaging across anti-diagonals:

$$\tilde{y}_k = \frac{1}{|A_k|} \sum_{(i,j) \in A_k} \tilde{X}_{ij}$$

**Where:**
*   $A_k = \{ (i,j) : i+j = k+1 \}$
*   $|A_k|$ is the number of elements in the set $A_k$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def run_ssa_denoise(series, L=14, keep_components=3):
    y = np.array(series, dtype=float)
    N = len(y)
    K = N - L + 1
    
    # 1. Embedding
    X = np.zeros((L, K))
    for i in range(K):
        X[:, i] = y[i : i + L]
        
    # 2. SVD
    U, S, V = np.linalg.svd(X, full_matrices=False)
    
    # 3. Grouping & Reconstruction
    r = min(keep_components, L, K)
    X_reconstructed = np.zeros((L, K))
    for i in range(r):
        X_reconstructed += S[i] * np.outer(U[:, i], V[i, :])
        
    # 4. Diagonal Averaging
    y_denoised = np.zeros(N)
    for i in range(N):
        j_min = max(0, i - K + 1)
        j_max = min(i, L - 1)
        count = 0
        for j in range(j_min, j_max + 1):
            y_denoised[i] += X_reconstructed[j, i - j]
            count += 1
        y_denoised[i] /= count
            
    return y_denoised

# Demonstration
t = np.arange(100)
signal = 0.5 * t + 15 * np.sin(2 * np.pi * t / 7)
noisy = signal + np.random.normal(0, 5, 100)
denoised = run_ssa_denoise(noisy)

plt.figure(figsize=(10, 5))
plt.plot(noisy, label='Raw Data', alpha=0.3)
plt.plot(denoised, label='SSA Denoised', color='red')
plt.title("SSA Implementation Check")
plt.legend()
plt.show()